**Trabajo de fin de master UNIR**


**Detección de lesiones oculares con YOLOv8**

In [ ]:
# Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Instalar dependencias
!pip install ultralytics opencv-python matplotlib seaborn scikit-learn

Importar librerias

In [ ]:
import cv2
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from ultralytics import YOLO
from sklearn.model_selection import KFold
import torch
import scipy.stats as stats

Definir rutas del dataset

In [ ]:
# Rutas de imágenes originales
train_images = "/content/drive/MyDrive/TFM-Imagenes/Deteccion/Original_Images/a. Training Set"
val_images   = "/content/drive/MyDrive/TFM-Imagenes/Deteccion/Original_Images/b. Testing Set"

# Rutas de máscaras
train_masks = {
    "microaneurisma": "/content/drive/MyDrive/TFM-Imagenes/Deteccion/All_Groundtruths/a. Training Set/1. Microaneurysms",
    "hemorragia": "/content/drive/MyDrive/TFM-Imagenes/Deteccion/All_Groundtruths/a. Training Set/2. Haemorrhages",
    "exudado_duro": "/content/drive/MyDrive/TFM-Imagenes/Deteccion/All_Groundtruths/a. Training Set/3. Hard Exudates",
    "exudado_blando": "/content/drive/MyDrive/TFM-Imagenes/Deteccion/All_Groundtruths/b. Testing Set/4. Soft Exudates",
    "disco_optico": "/content/drive/MyDrive/TFM-Imagenes/Deteccion/All_Groundtruths/a. Training Set/5. Optic Disc"
}

val_masks = {
    "microaneurisma": "/content/drive/MyDrive/TFM-Imagenes/Deteccion/All_Groundtruths/b. Testing Set/1. Microaneurysms",
    "hemorragia": "/content/drive/MyDrive/TFM-Imagenes/Deteccion/All_Groundtruths/b. Testing Set/2. Haemorrhages",
    "exudado_duro": "/content/drive/MyDrive/TFM-Imagenes/Deteccion/All_Groundtruths/b. Testing Set/3. Hard Exudates",
    "exudado_blando": "/content/drive/MyDrive/TFM-Imagenes/Deteccion/All_Groundtruths/b. Testing Set/4. Soft Exudates",
    "disco_optico": "/content/drive/MyDrive/TFM-Imagenes/Deteccion/All_Groundtruths/b. Testing Set/5. Optic Disc"
}


Crear estructura YOLO

In [ ]:
# Crear carpetas YOLO
!mkdir -p /content/drive/MyDrive/TFM-Imagenes/YOLO-IDRiD/images/train
!mkdir -p /content/drive/MyDrive/TFM-Imagenes/YOLO-IDRiD/images/val
!mkdir -p /content/drive/MyDrive/TFM-Imagenes/YOLO-IDRiD/labels/train
!mkdir -p /content/drive/MyDrive/TFM-Imagenes/YOLO-IDRiD/labels/val

# Copiar imágenes
!cp "$train_images"/*.jpg /content/drive/MyDrive/TFM-Imagenes/YOLO-IDRiD/images/train/
!cp "$val_images"/*.jpg /content/drive/MyDrive/TFM-Imagenes/YOLO-IDRiD/images/val/

Conversión de máscaras a bounding boxes YOLO

In [ ]:
import os

# Listar primeras imágenes y máscaras
print("Ejemplos de imágenes en entrenamiento:")
train_imgs = sorted([f for f in os.listdir(train_images) if f.endswith(".jpg")])
print(train_imgs[:10])  # muestra los primeros 10

print("\nEjemplos de máscaras de microaneurismas:")
ma_masks = sorted([f for f in os.listdir(train_masks["microaneurisma"]) if f.endswith(".tif")])
print(ma_masks[:10])  # muestra los primeros 10

print("\nEjemplos de máscaras de hemorragias:")
he_masks = sorted([f for f in os.listdir(train_masks["hemorragia"]) if f.endswith(".tif")])
print(he_masks[:10])

In [ ]:
import cv2
import os

# Diccionario de clases con sufijos correctos (solo 3 clases)
classes = {
    "microaneurisma": ("_MA.tif", 0),
    "hemorragia": ("_HE.tif", 1),
    "exudado_duro": ("_EX.tif", 2)
}

def mask_to_yolo(mask_path, class_id, img_w, img_h):
    mask = cv2.imread(mask_path, 0)
    if mask is None:
        return []
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    yolo_labels = []
    for cnt in contours:
        x,y,w,h = cv2.boundingRect(cnt)
        if w > 0 and h > 0:
            x_center = (x + w/2) / img_w
            y_center = (y + h/2) / img_h
            width = w / img_w
            height = h / img_h
            yolo_labels.append(f"{class_id} {x_center} {y_center} {width} {height}")
    return yolo_labels

def generar_labels(img_dir, label_dir, masks_dict, masks_root):
    os.makedirs(label_dir, exist_ok=True)
    count_labels = 0
    for img_file in os.listdir(img_dir):
        if not img_file.endswith(".jpg"):
            continue
        img_path = os.path.join(img_dir, img_file)
        img = cv2.imread(img_path)
        if img is None:
            continue
        h, w, _ = img.shape
        label_lines = []
        base_name = img_file.replace(".jpg", "")
        for cls_name, (suffix, cls_id) in masks_dict.items():
            mask_dir = masks_root.get(cls_name)
            if mask_dir:
                mask_path = os.path.join(mask_dir, base_name + suffix)
                if os.path.exists(mask_path):
                    label_lines.extend(mask_to_yolo(mask_path, cls_id, w, h))
        if label_lines:
            with open(os.path.join(label_dir, img_file.replace(".jpg", ".txt")), "w") as f:
                f.write("\n".join(label_lines))
            count_labels += 1
    print(f"Se generaron {count_labels} archivos de etiquetas en {label_dir}")

# Directorios de etiquetas
train_label_dir = "/content/drive/MyDrive/TFM-Imagenes/YOLO-IDRiD/labels/train"
val_label_dir   = "/content/drive/MyDrive/TFM-Imagenes/YOLO-IDRiD/labels/val"

# Diccionarios de máscaras (ajusta rutas según tu Drive)
train_masks = {
    "microaneurisma": "/content/drive/MyDrive/TFM-Imagenes/Deteccion/All_Groundtruths/a. Training Set/1. Microaneurysms",
    "hemorragia": "/content/drive/MyDrive/TFM-Imagenes/Deteccion/All_Groundtruths/a. Training Set/2. Haemorrhages",
    "exudado_duro": "/content/drive/MyDrive/TFM-Imagenes/Deteccion/All_Groundtruths/a. Training Set/3. Hard Exudates"
}

val_masks = {
    "microaneurisma": "/content/drive/MyDrive/TFM-Imagenes/Deteccion/All_Groundtruths/b. Testing Set/1. Microaneurysms",
    "hemorragia": "/content/drive/MyDrive/TFM-Imagenes/Deteccion/All_Groundtruths/b. Testing Set/2. Haemorrhages",
    "exudado_duro": "/content/drive/MyDrive/TFM-Imagenes/Deteccion/All_Groundtruths/b. Testing Set/3. Hard Exudates"
}

# Generar etiquetas para train y val
generar_labels(train_images, train_label_dir, classes, train_masks)
generar_labels(val_images, val_label_dir, classes, val_masks)

Nuevo dataset para obtener mas datos DDR

In [ ]:
import cv2
import os

# Mapeo de clases
classes = {"MA":0, "HE":1, "EX":2, "SE":3}

def mask_to_yolo(mask_path, class_id, img_w, img_h):
    mask = cv2.imread(mask_path, 0)
    if mask is None:
        return []
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    yolo_labels = []
    for cnt in contours:
        x,y,w,h = cv2.boundingRect(cnt)
        if w > 0 and h > 0:
            x_center = (x + w/2) / img_w
            y_center = (y + h/2) / img_h
            width = w / img_w
            height = h / img_h
            yolo_labels.append(f"{class_id} {x_center} {y_center} {width} {height}")
    return yolo_labels

def generar_labels(img_dir, label_dirs, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    for img_file in os.listdir(img_dir):
        if not img_file.endswith(".jpg"):
            continue
        img_path = os.path.join(img_dir, img_file)
        img = cv2.imread(img_path)
        if img is None:
            continue
        h, w, _ = img.shape
        label_lines = []
        base_name = img_file.replace(".jpg", "")
        for cls_name, cls_id in classes.items():
            mask_dir = label_dirs.get(cls_name)
            if mask_dir:
                mask_path = os.path.join(mask_dir, base_name + ".tif")
                if os.path.exists(mask_path):
                    label_lines.extend(mask_to_yolo(mask_path, cls_id, w, h))
        if label_lines:
            with open(os.path.join(out_dir, base_name + ".txt"), "w") as f:
                f.write("\n".join(label_lines))

# Ejemplo: generar etiquetas para train
train_img_dir = r"/content/drive/MyDrive/TFM-Imagenes/Deteccion/DDR/lesion_segmentation/train/image"
train_label_dirs = {
    "EX": r"/content/drive/MyDrive/TFM-Imagenes/Deteccion/DDR/lesion_segmentation/train/label/EX",
    "HE": r"/content/drive/MyDrive/TFM-Imagenes/Deteccion/DDR/lesion_segmentation/train/label/HE",
    "MA": r"/content/drive/MyDrive/TFM-Imagenes/Deteccion/DDR/lesion_segmentation/train/label/MA",
    "SE": r"/content/drive/MyDrive/TFM-Imagenes/Deteccion/DDR/lesion_segmentation/train/label/SE"
}
out_train = r"/content/drive/MyDrive/TFM-Imagenes/YOLO-IDRiD/labels/train"

generar_labels(train_img_dir, train_label_dirs, out_train)

# Generar etiquetas para validación
val_img_dir = r"/content/drive/MyDrive/TFM-Imagenes/Deteccion/DDR/lesion_segmentation/valid/image"
val_label_dirs = {
    "EX": r"/content/drive/MyDrive/TFM-Imagenes/Deteccion/DDR/lesion_segmentation/valid/segmentation label/EX",
    "HE": r"/content/drive/MyDrive/TFM-Imagenes/Deteccion/DDR/lesion_segmentation/valid/segmentation label/HE",
    "MA": r"/content/drive/MyDrive/TFM-Imagenes/Deteccion/DDR/lesion_segmentation/valid/segmentation label/MA",
    "SE": r"/content/drive/MyDrive/TFM-Imagenes/Deteccion/DDR/lesion_segmentation/valid/segmentation label/SE"
}
out_val = r"/content/drive/MyDrive/TFM-Imagenes/YOLO-IDRiD/labels/val"

generar_labels(val_img_dir, val_label_dirs, out_val)


Etiquetas generadas

In [ ]:
import os

# Directorios de etiquetas
train_labels = r"/content/drive/MyDrive/TFM-Imagenes/YOLO-IDRiD/labels/train"
val_labels   = r"/content/drive/MyDrive/TFM-Imagenes/YOLO-IDRiD/labels/val"

# Contar archivos .txt
train_count = len([f for f in os.listdir(train_labels) if f.endswith(".txt")])
val_count   = len([f for f in os.listdir(val_labels) if f.endswith(".txt")])

print(f"Total de etiquetas generadas en TRAIN: {train_count}")
print(f"Total de etiquetas generadas en VAL:   {val_count}")

Imágenes con máscara

In [ ]:
###cuantas imagenes tienen mascara

import os

val_img_dir = r"/content/drive/MyDrive/TFM-Imagenes/Deteccion/DDR/lesion_segmentation/valid/image"
val_mask_dirs = {
    "EX": r"/content/drive/MyDrive/TFM-Imagenes/Deteccion/DDR/lesion_segmentation/valid/segmentation label/EX",
    "HE": r"/content/drive/MyDrive/TFM-Imagenes/Deteccion/DDR/lesion_segmentation/valid/segmentation label/HE",
    "MA": r"/content/drive/MyDrive/TFM-Imagenes/Deteccion/DDR/lesion_segmentation/valid/segmentation label/MA",
    "SE": r"/content/drive/MyDrive/TFM-Imagenes/Deteccion/DDR/lesion_segmentation/valid/segmentation label/SE"
}

count_masks = 0
for img_file in os.listdir(val_img_dir):
    base_name = img_file.replace(".jpg", "")
    for cls, path in val_mask_dirs.items():
        mask_path = os.path.join(path, base_name + ".tif")
        if os.path.exists(mask_path):
            count_masks += 1
            break  # al menos una máscara encontrada
print(f"Imágenes con al menos una máscara en VALID: {count_masks}")

 El bloque de oversampling


In [ ]:
import os, shutil

train_img_dir = "/content/drive/MyDrive/TFM-Imagenes/YOLO-IDRiD/images/train"
train_lbl_dir = "/content/drive/MyDrive/TFM-Imagenes/YOLO-IDRiD/labels/train"

dup_count = 0

for lbl_file in os.listdir(train_lbl_dir):
    if not lbl_file.endswith(".txt"):
        continue
    lbl_path = os.path.join(train_lbl_dir, lbl_file)
    with open(lbl_path, "r") as f:
        lines = f.readlines()
    # Buscar clase MA (id=0)
    if any(line.startswith("0 ") for line in lines):
        base_name = lbl_file.replace(".txt", "")
        img_file = base_name + ".jpg"
        img_path = os.path.join(train_img_dir, img_file)
        if os.path.exists(img_path):
            # Crear tres duplicados adicionales (dup1, dup2, dup3)
            for i in range(1, 4):
                new_img = f"{base_name}_dup{i}.jpg"
                new_lbl = f"{base_name}_dup{i}.txt"
                shutil.copy(img_path, os.path.join(train_img_dir, new_img))
                shutil.copy(lbl_path, os.path.join(train_lbl_dir, new_lbl))
                dup_count += 1

print(f"Se cuadruplicaron {dup_count} imágenes con MA para oversampling")

Conteo de anotaciones por clase

In [ ]:
import os

train_label_dir = "/content/drive/MyDrive/TFM-Imagenes/YOLO-IDRiD/labels/train"
val_label_dir   = "/content/drive/MyDrive/TFM-Imagenes/YOLO-IDRiD/labels/val"

conteo_clases = {0:0, 1:0, 2:0, 3:0, 4:0}

for label_dir in [train_label_dir, val_label_dir]:
    for file in os.listdir(label_dir):
        if file.endswith(".txt"):
            with open(os.path.join(label_dir, file)) as f:
                for line in f:
                    cls = int(line.split()[0])
                    conteo_clases[cls] += 1

print("Conteo de anotaciones por clase:")
for cls_id, count in conteo_clases.items():
    print(f"Clase {cls_id}: {count} anotaciones")

Crear archivo data.yaml

In [ ]:
data_yaml = """
train: /content/drive/MyDrive/TFM-Imagenes/YOLO-IDRiD/images/train
val: /content/drive/MyDrive/TFM-Imagenes/YOLO-IDRiD/images/val
test: /content/drive/MyDrive/TFM-Imagenes/YOLO-IDRiD/images/test
nc: 4
names: ["MA", "HE", "EX", "SE"]
"""

with open("data.yaml", "w") as f:
    f.write(data_yaml)

Entrenamiento YOLOv8 con Early Stopping









In [ ]:
model = YOLO("yolov8n.pt")

results_train = model.train(
    data="data.yaml",
    epochs=100,
    patience=50,
    imgsz=640,
    batch=32,
    augment=True,        # activa augmentations
    degrees=10,          # rotación aleatoria
    translate=0.1,       # traslación
    scale=0.9,           # zoom aleatorio
    shear=1.0,           # deformación
    flipud=0.0,          # flip vertical (0 = desactivado)
    fliplr=0.5,          # flip horizontal (50% probabilidad)
    mosaic=1.0,          # mezcla de 4 imágenes
    mixup=0.2,           # mezcla de imágenes
    hsv_h=0.015,         # variación de tono
    hsv_s=0.7,           # variación de saturación
    hsv_v=0.4,            # variación de brillo
    optimizer="AdamW",
    lr0=0.01,
    lrf=0.1,
    warmup_epochs=3

)

Validación cruzada k-fold

In [ ]:
from sklearn.model_selection import KFold
import pandas as pd

kf = KFold(n_splits=3, shuffle=True, random_state=42)
fold_metrics = []

image_files = os.listdir(train_images)

for fold, (train_idx, val_idx) in enumerate(kf.split(image_files)):
    print(f"Fold {fold+1}")

    # Crear listas de train y val para este fold
    train_split = [image_files[i] for i in train_idx]
    val_split   = [image_files[i] for i in val_idx]

    # Entrenar modelo desde cero en este fold
    model = YOLO("yolov8n.pt")  # carga modelo base
    model.train(
        data="data.yaml",
        epochs=100,
        batch=16,
        augment=True,
        degrees=10,
        fliplr=0.5,
        scale=0.5
    )

    # Validar en este fold
    r = model.val()
    metrics = r.results_dict

    fold_metrics.append({
        "fold": fold+1,
        "precision": metrics.get("precision", 0),
        "recall": metrics.get("recall", 0),
        "f1": metrics.get("f1", 0),
        "mAP50": metrics.get("mAP50", 0),
        "mAP50-95": metrics.get("mAP50-95", 0)
    })

fold_df = pd.DataFrame(fold_metrics)
print(fold_df)

Distribución de clases

In [ ]:
label_dir = "/content/drive/MyDrive/TFM-Imagenes/YOLO-IDRiD/labels/train"
class_counts = {}
for file in os.listdir(label_dir):
    with open(os.path.join(label_dir, file)) as f:
        for line in f:
            cls = int(line.split()[0])
            class_counts[cls] = class_counts.get(cls, 0) + 1

sns.barplot(x=list(class_counts.keys()), y=list(class_counts.values()))
plt.title("Distribución de clases en el dataset")
plt.xlabel("Clase")
plt.ylabel("Número de ejemplos")
plt.show()

ANOVA

In [ ]:
from scipy.stats import f_oneway

# Convertir columnas a listas
f1_scores = fold_df["f1"].tolist()
precision_scores = fold_df["precision"].tolist()
iou_scores = fold_df["mAP50"].tolist()  # puedes usar IoU o mAP según lo que quieras comparar

# ANOVA entre métricas
anova_result = f_oneway(f1_scores, precision_scores, iou_scores)
print("ANOVA F-statistic:", anova_result.statistic)
print("ANOVA p-value:", anova_result.pvalue)

# Interpretación rápida
if anova_result.pvalue < 0.05:
    print("✅ Diferencias estadísticamente significativas entre las métricas.")
else:
    print("⚠️ No se encontraron diferencias significativas.")

In [ ]:
from scipy.stats import f_oneway

f1_scores = [x for x in fold_df["f1"].tolist() if x > 0]
precision_scores = [x for x in fold_df["precision"].tolist() if x > 0]
iou_scores = [x for x in fold_df["mAP50"].tolist() if x > 0]

if len(f1_scores) > 1 and len(precision_scores) > 1 and len(iou_scores) > 1:
    anova_result = f_oneway(f1_scores, precision_scores, iou_scores)
    print("ANOVA F-statistic:", anova_result.statistic)
    print("ANOVA p-value:", anova_result.pvalue)
else:
    print("⚠️ No hay suficiente variación en los datos para calcular ANOVA.")

Verificación etiquetas YOLO

In [ ]:
###Verificacion de etiquetas YOLO

import os

# Directorios de etiquetas
train_label_dir = "/content/drive/MyDrive/TFM-Imagenes/YOLO-IDRiD/labels/train"
val_label_dir   = "/content/drive/MyDrive/TFM-Imagenes/YOLO-IDRiD/labels/val"

def verificar_labels(label_dir):
    vacios = []
    con_contenido = 0
    for file in os.listdir(label_dir):
        if not file.endswith(".txt"):
            continue
        with open(os.path.join(label_dir, file)) as f:
            contenido = f.read().strip()
            if contenido == "":
                vacios.append(file)
            else:
                con_contenido += 1
    print(f"En {label_dir}:")
    print(f" - Archivos con contenido: {con_contenido}")
    print(f" - Archivos vacíos: {len(vacios)}")
    if vacios:
        print("Ejemplos de archivos vacíos:", vacios[:10])

# Verificar entrenamiento y validación
verificar_labels(train_label_dir)
verificar_labels(val_label_dir)

Visualizar predicciones

In [ ]:
import cv2
import matplotlib.pyplot as plt

# Directorio de imágenes y etiquetas
val_img_dir = "/content/drive/MyDrive/TFM-Imagenes/YOLO-IDRiD/images/val"
val_label_dir = "/content/drive/MyDrive/TFM-Imagenes/YOLO-IDRiD/labels/val"

# Nombres de clases (solo 3)
class_names = ["microaneurisma", "hemorragia", "exudado_duro"]

# Selecciona una imagen de validación
img_name = "IDRiD_55.jpg"  # cámbiala por otra si quieres
img_path = os.path.join(val_img_dir, img_name)
label_path = os.path.join(val_label_dir, img_name.replace(".jpg", ".txt"))

# Cargar imagen
img = cv2.imread(img_path)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
h, w, _ = img.shape

# Dibujar bounding boxes desde el .txt
if os.path.exists(label_path):
    with open(label_path) as f:
        for line in f:
            cls, x_center, y_center, bw, bh = map(float, line.split())
            cls = int(cls)
            x1 = int((x_center - bw/2) * w)
            y1 = int((y_center - bh/2) * h)
            x2 = int((x_center + bw/2) * w)
            y2 = int((y_center + bh/2) * h)
            cv2.rectangle(img, (x1,y1), (x2,y2), (255,0,0), 2)
            cv2.putText(img, class_names[cls], (x1,y1-5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,0,0), 2)

# Mostrar imagen con anotaciones
plt.figure(figsize=(10,10))
plt.imshow(img)
plt.axis("off")
plt.show()


**Ejecución del modelo ya entrenado**

Comparar anotaciones y predicciones.

El modelo entrenado se encuentra en la raiz del repositorio con el nombre de DeteccionYOLOv8.


In [ ]:
import cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO
import os

# Directorios
val_img_dir = "/content/drive/MyDrive/TFM-Imagenes/YOLO-IDRiD/images/val"
val_label_dir = "/content/drive/MyDrive/TFM-Imagenes/YOLO-IDRiD/labels/val"

# Nombres de clases (ajustado a las 4 clases de data.yaml)
class_names = ['MA', 'HE', 'EX', 'SE']
# Definir colores por clase para la visualización
class_colors = {
    0: (255, 0, 0),    # Red for MA
    1: (0, 255, 0),    # Green for HE
    2: (0, 0, 255),    # Blue for EX
    3: (255, 255, 0)   # Yellow for SE
}

# Selecciona una imagen de validación
img_name = "IDRiD_70.jpg"  # cámbiala por otra si quieres
img_path = os.path.join(val_img_dir, img_name)
label_path = os.path.join(val_label_dir, img_name.replace(".jpg", ".txt"))

# Cargar imagen
img = cv2.imread(img_path)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
h, w, _ = img.shape

# Copias para anotaciones y predicciones
img_ann = img.copy()
img_pred = img.copy()

# Dibujar anotaciones (ground truth)
if os.path.exists(label_path):
    with open(label_path) as f:
        for line in f:
            cls, x_center, y_center, bw, bh = map(float, line.split())
            cls = int(cls)
            x1 = int((x_center - bw/2) * w)
            y1 = int((y_center - bh/2) * h)
            x2 = int((x_center + bw/2) * w)
            y2 = int((y_center + bh/2) * h)
            # Usar class_colors para el ground truth
            color = class_colors.get(cls, (255, 255, 255)) # Default a blanco si la clase no está en el diccionario
            cv2.rectangle(img_ann, (x1,y1), (x2,y2), color, 2)
            cv2.putText(img_ann, class_names[cls], (x1,y1-5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

# Cargar modelo entrenado y hacer predicción
model = YOLO("runs/detect/train/weights/best.pt")  # ajusta la ruta a tu modelo entrenado
results = model.predict(source=img_path, conf=0.25) # Reducir umbral de confianza

# Dibujar predicciones
for r in results:
    for box in r.boxes:
        cls = int(box.cls[0])
        if cls < len(class_names):   # ✅ evita IndexError
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            # Usar class_colors para las predicciones
            color = class_colors.get(cls, (255, 255, 255)) # Default a blanco si la clase no está en el diccionario
            cv2.rectangle(img_pred, (x1,y1), (x2,y2), color, 2)
            cv2.putText(img_pred, class_names[cls], (x1,y1-5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

# Mostrar lado a lado
plt.figure(figsize=(20,10))
plt.subplot(1,2,1)
plt.title("Anotaciones (Ground Truth)")
plt.imshow(img_ann)
plt.axis("off")

plt.subplot(1,2,2)
plt.title("Predicciones del Modelo")
plt.imshow(img_pred)
plt.axis("off")
plt.show()

**Tabla de resultados de detección de enfermedades**

In [ ]:
# Tabla de deteccion de enfermedades
import pandas as pd

detected_class_counts = {}
for r in results:
    for box in r.boxes:
        cls_id = int(box.cls[0])
        class_name = class_names[cls_id] if cls_id < len(class_names) else f"Unknown Class {cls_id}"
        detected_class_counts[class_name] = detected_class_counts.get(class_name, 0) + 1

# Prepare data for DataFrame
table_data = []
for cls_id, name in enumerate(class_names):
    color = class_colors.get(cls_id, (0, 0, 0)) # Default to black if not defined
    color_str = f"RGB{color}"
    count = detected_class_counts.get(name, 0)
    table_data.append({"ID Clase": cls_id, "Nombre Clase": name, "Color (RGB)": color_str, "Cantidad Detectada": count})

df_detections = pd.DataFrame(table_data)
print("\nTabla de Detecciones por Clase:")
display(df_detections)

Correlación entre métricas

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Selecciona solo las columnas que existen en tu DataFrame
corr = fold_df[["precision","recall","f1","mAP50","mAP50-95"]].corr()

sns.heatmap(corr, annot=True, cmap="coolwarm")
plt.title("Correlación entre métricas")
plt.show()

Grad-CAM adaptado a YOLOv8

In [ ]:
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

def grad_cam_yolo(model, img_path, target_layer_index=10):
    """
    Aplica Grad-CAM sobre YOLOv8 en la capa indicada.
    Args:
        model: modelo YOLOv8 cargado
        img_path: ruta de la imagen
        target_layer_index: índice de la capa en model.model.model donde enganchar los hooks
    """
    # Mostrar capas disponibles
    print("Capas del modelo:")
    for i, layer in enumerate(model.model.model):
        print(i, layer)

    # Cargar imagen
    img = cv2.imread(img_path)
    if img is None:
        raise FileNotFoundError(f"No se pudo cargar la imagen en {img_path}")
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Convertir a tensor normalizado y asegurar que requiere gradientes
    img_tensor = torch.from_numpy(img_rgb).float().permute(2,0,1).unsqueeze(0) / 255.0
    img_tensor.requires_grad_(True) # <--- Fix: Set requires_grad to True

    # Diccionarios para activaciones y gradientes
    activations, gradients = {}, {}

    def forward_hook(module, input, output):
        activations["value"] = output

    def backward_hook(module, grad_in, grad_out):
        gradients["value"] = grad_out[0]

    # Registrar hooks en la capa objetivo
    layer = model.model.model[target_layer_index]
    layer.register_forward_hook(forward_hook)
    layer.register_backward_hook(backward_hook)

    # Forward pass
    # Asegúrate de que el modelo esté en modo de evaluación y que los gradientes estén habilitados
    model.eval() # Poner el modelo en modo de evaluación
    with torch.enable_grad(): # Habilitar gradientes explícitamente para esta sección
        preds = model(img_tensor, verbose=False)

    # Tomar la predicción más confiable
    if len(preds[0].boxes) == 0:
        print("⚠️ No se detectaron objetos en la imagen.")
        return
    score = preds[0].boxes.conf.max()
    score.backward() # Ahora esto debería funcionar

    # Grad-CAM
    grads = gradients["value"]
    acts = activations["value"]
    weights = grads.mean(dim=(2,3), keepdim=True)
    cam = (weights * acts).sum(dim=1).squeeze().detach().cpu().numpy()
    cam = np.maximum(cam, 0)
    cam = cv2.resize(cam, (img_rgb.shape[1], img_rgb.shape[0]))
    cam = cam / cam.max()

    # Superponer heatmap
    heatmap = cv2.applyColorMap(np.uint8(255*cam), cv2.COLORMAP_JET)
    superimposed = cv2.addWeighted(img_rgb, 0.6, heatmap, 0.4, 0)

    # Mostrar resultado
    plt.figure(figsize=(10,10))
    plt.imshow(superimposed)
    plt.axis("off")
    plt.title(f"Grad-CAM sobre capa {target_layer_index}")
    plt.show()

# 🔹 Ejemplo de uso
model = YOLO("runs/detect/train/weights/best.pt")  # ajusta la ruta a tu modelo entrenado
grad_cam_yolo(model, "/content/drive/MyDrive/TFM-Imagenes/YOLO-IDRiD/images/val/IDRiD_55.jpg", target_layer_index=10)

Graficas de resultados

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import os
import scipy.stats as stats
import seaborn as sns

# ============================================
# 1. Distribución de clases
# ============================================

class_names_list = ["MA", "HE", "EX", "SE"]

train_label_dir = "/content/drive/MyDrive/TFM-Imagenes/YOLO-IDRiD/labels/train"
val_label_dir   = "/content/drive/MyDrive/TFM-Imagenes/YOLO-IDRiD/labels/val"

conteo_clases_actual = {i: 0 for i in range(len(class_names_list))}

for label_dir in [train_label_dir, val_label_dir]:
    if os.path.exists(label_dir):
        for file in os.listdir(label_dir):
            if file.endswith(".txt"):
                with open(os.path.join(label_dir, file)) as f:
                    for line in f:
                        try:
                            cls = int(line.split()[0])
                            if cls in conteo_clases_actual:
                                conteo_clases_actual[cls] += 1
                        except (ValueError, IndexError):
                            # Skip malformed lines
                            continue
    else:
        print(f"Warning: Directorio no encontrado: {label_dir}")

filtered_class_counts = {class_names_list[cls_id]: count for cls_id, count in conteo_clases_actual.items() if count > 0 and cls_id < len(class_names_list)}

df_classes = pd.DataFrame(list(filtered_class_counts.items()), columns=["Clase", "Instancias"])
df_classes["Proporción (%)"] = df_classes["Instancias"] / df_classes["Instancias"].sum() * 100

print("Tabla de distribución de clases:")
print(df_classes)

# Gráfico de barras
plt.figure(figsize=(8,5))
plt.bar(df_classes["Clase"], df_classes["Instancias"], color=["#1f77b4","#ff7f0e","#2ca02c", "#d62728"][:len(df_classes)])
plt.title("Distribución de clases en el dataset (Real)")
plt.xlabel("Clase")
plt.ylabel("Número de instancias")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


# ============================================
# 2. ANOVA (comparación entre métricas del K-Fold)
# ============================================

if 'fold_df' in locals() and not fold_df.empty:
    f1_scores = [x for x in fold_df["f1"].tolist() if x > 0]
    precision_scores = [x for x in fold_df["precision"].tolist() if x > 0]
    iou_scores = [x for x in fold_df["mAP50"].tolist() if x > 0]

    if len(f1_scores) > 1 and len(precision_scores) > 1 and len(iou_scores) > 1:
        anova_result = stats.f_oneway(f1_scores, precision_scores, iou_scores)
        print("\nANOVA F-statistic (comparando F1, Precision, mAP50 entre folds):", round(anova_result.statistic, 3))
        print("p-value:", round(anova_result.pvalue, 4))
        if anova_result.pvalue < 0.05:
            print("✅ Diferencias estadísticamente significativas entre las métricas en los diferentes folds.")
        else:
            print("⚠️ No se encontraron diferencias significativas entre las métricas en los diferentes folds.")
    else:
        print("⚠️ No hay suficiente variación o datos válidos en fold_df para calcular ANOVA entre F1, Precision e mAP50.")

    mean_metrics = fold_df[["precision", "recall", "f1", "mAP50", "mAP50-95"]].mean()
    std_metrics = fold_df[["precision", "recall", "f1", "mAP50", "mAP50-95"]].std()
    df_anova_summary = pd.DataFrame([mean_metrics, std_metrics], index=["Media", "Desv. Estándar"])
    print("\nTabla Resumen de Métricas del K-Fold:")
    print(df_anova_summary)

else:
    print("\n⚠️ 'fold_df' no está disponible o está vacío. Ejecute la celda de validación cruzada k-fold (WV-d4rVlARTP) primero.")


# ============================================
# 3. Correlación entre métricas
# ============================================

if 'fold_df' in locals() and not fold_df.empty:
    corr = fold_df[["precision","recall","f1","mAP50","mAP50-95"]].corr()

    print("\nTabla de correlación entre métricas:")
    print(corr)

    plt.figure(figsize=(6,4))
    sns.heatmap(corr, annot=True, cmap="coolwarm", vmin=-1, vmax=1)
    plt.title("Correlación entre métricas (Real)")
    plt.show()
else:
    print("\n⚠️ 'fold_df' no está disponible o está vacío. No se puede calcular la correlación.")

### Guardar el modelo entrenado y los archivos generados en Google Drive

Esta celda copiará el directorio completo de resultados del último entrenamiento (`runs/detect/train`) a la ruta especificada en Google Drive. Esto incluye el modelo `best.pt`, `last.pt`, logs, gráficos y cualquier otro archivo generado.

In [ ]:
import shutil
import os

# Directorio de origen del último entrenamiento de YOLOv8
source_dir = "/content/runs/detect/train" # Por defecto, Ultralytics guarda los resultados aquí

# Directorio de destino en Google Drive
dest_dir = "/content/drive/MyDrive/TFM-Imagenes/Modelo_Entrenado_Final_TFM"

# Crear el directorio de destino si no existe
os.makedirs(dest_dir, exist_ok=True)

# Copiar el contenido del directorio de origen al destino
# Si el directorio de destino ya existe, copytree puede dar error,
# por lo que primero eliminamos el destino si ya existe o manejamos el error.
if os.path.exists(os.path.join(dest_dir, os.path.basename(source_dir))):
    print(f"Eliminando el directorio existente en el destino: {os.path.join(dest_dir, os.path.basename(source_dir))}")
    shutil.rmtree(os.path.join(dest_dir, os.path.basename(source_dir)))

print(f"Copiando resultados de '{source_dir}' a '{dest_dir}'...")
shutil.copytree(source_dir, os.path.join(dest_dir, os.path.basename(source_dir)))
print("¡Copia completada con éxito!")


Copiando resultados de '/content/runs/detect/train' a '/content/drive/MyDrive/TFM-Imagenes/Modelo_Entrenado_Final_TFM'...
¡Copia completada con éxito!
